Intrusion Detection System based on Random Forest


importing Libraries

In [ ]:
# importing required libraries
import numpy as np
import pandas as pd 
import pickle # saving and loading trained model
from os import path

# importing required libraries for normalizing data
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler,LabelBinarizer
from sklearn.ensemble import RandomForestClassifier

# importing library for plotting
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report, confusion_matrix,ConfusionMatrixDisplay

from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models

import keras
from keras.applications import VGG16
from keras.layers import Dense, LSTM, MaxPool1D, Flatten, Dropout # importing dense layer
from keras.models import Sequential #importing Sequential layer
from keras.layers import Input
from keras.models import Model
# representation of model layers

In [ ]:
feature=["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent","hot",
          "num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations","num_shells",
          "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
          "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count", 
          "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
          "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]

Importing Dataset

In [ ]:
train_data = pd.read_csv("Dataset/KDDTrain+.txt", names=feature)
test_data=pd.read_csv("Dataset/KDDTest+.txt", names=feature)
test_data21=pd.read_csv("Dataset/KDDTest-21.txt", names=feature)
data=pd.concat([train_data,test_data],ignore_index=True)

In [ ]:
data.drop(["difficulty"],axis=1,inplace=True)

In [ ]:
data.info()

In [ ]:
data['label'].value_counts()

Changing Labels

In [ ]:
def label_change(df):
    df.label.replace(['apache2','back','land','neptune','mailbomb','pod','processtable','smurf','teardrop','udpstorm','worm'],'Dos',inplace=True)
    df.label.replace(['ftp_write','guess_passwd','imap','multihop','phf','spy','warezclient','warezmaster','snmpgetattack','httptunnel','snmpguess','sendmail','named','xlock','xsnoop'],'R2L',inplace=True)
    df.label.replace(['ipsweep','nmap','portsweep','satan','mscan','saint'],'Probe',inplace=True)
    df.label.replace(['buffer_overflow','loadmodule','perl','rootkit','ps','xterm','sqlattack'],'U2R',inplace=True)

In [ ]:
label_change(data)

In [ ]:
data.label.value_counts()

Dataframe

In [ ]:
label = pd.DataFrame(data.label)

In [ ]:
scaler=StandardScaler();
def standardization(data,col):
    for i in col:
        arr=data[i]
        arr=np.array(arr)
        data[i]=scaler.fit_transform(arr.reshape(len(arr),1))
    return data
numeric_col=data.select_dtypes(include='number').columns
data=standardization(data,numeric_col)

In [ ]:
le=preprocessing.LabelEncoder()
encode=label.apply(le.fit_transform)
data['intrusion']=encode


In [ ]:
data.drop(['label'],axis=1,inplace=True)

In [ ]:
data=pd.get_dummies(data,columns=['protocol_type','service','flag'],prefix="",prefix_sep="")
print(data.shape)

In [ ]:
y_data=data['intrusion']
X_data=data.drop(labels=['intrusion'],axis=1)

print('X_train has shape:',X_data.shape,'\nY_train has shape:',y_data.shape)

In [ ]:
lb=LabelBinarizer()
lb.fit(y_data)
X_data=np.array(X_data)
y_data=np.array(y_data)


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)
print(X_train.shape,'\n',X_test.shape)

In [ ]:
model=RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X_train,Y_train)

In [ ]:
Y_pred=model.predict(X_test)
print(classification_report(Y_test,Y_pred))

In [ ]:
print('Accuracy:',accuracy_score(Y_test,Y_pred))

In [ ]:
Y_test = Y_test.reshape(-1, 1)
Y_pred = Y_pred.reshape(-1, 1)

In [ ]:
Y_test_org=lb.inverse_transform(Y_test)
Y_pred_org=lb.inverse_transform(Y_pred)
cm=confusion_matrix(Y_test_org,Y_pred_org)
print(cm)

In [ ]:
cm_display = ConfusionMatrixDisplay(cm)
cm_display.plot()

In [ ]:
train_acc = model.score(X_train, Y_train)
test_acc = accuracy_score(Y_test, Y_pred)

In [ ]:
print('Training Accuracy:', train_acc)
print('Testing Accuracy:', test_acc)

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(['Training', 'Testing'], [train_acc, test_acc])
plt.ylabel('Accuracy')
plt.title('Training and Testing Accuracy')
plt.show()